# Session 13 — Deep learning for Earth observation (EuroSAT)
*Séance 13 — Apprentissage profond pour l'OT (EuroSAT)*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noe2019/ML4EO_Bootcamp/blob/main/notebooks/13_apprentissage_profond_eurosat.ipynb)

**Course:** Machine Learning for Earth Observation (45 h) · **Instructor:** Noé Carème Fouotsa Manfouo

**By the end of this session you will be able to:**

1. Explain how convolutional neural networks (CNNs) learn spatial patterns from image patches.
2. Fine-tune an ImageNet-pretrained ResNet-18 on the EuroSAT land-use dataset (transfer learning).
3. Evaluate a deep model with a confusion matrix and inspect its errors.
4. Find open EO training datasets (Source Cooperative, TorchGeo) now that Radiant MLHub has closed.

**How to use this notebook:** read the lecture cells, run every code cell (`Shift + Enter`), then complete the cells marked
`# YOUR CODE HERE` in the lab section. Run the ✅ check cells to get instant feedback.

In [ ]:
# --- Setup (run once) -------------------------------------------------------
import sys, pathlib, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not pathlib.Path("ML4EO_Bootcamp").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/noe2019/ML4EO_Bootcamp.git"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *"geopandas rasterio rioxarray torchvision".split()], check=True)
    ROOT = pathlib.Path("ML4EO_Bootcamp")
else:
    ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
DATA = ROOT / "data"
sys.path.append(str(ROOT / "notebooks"))   # course helper module (ml4eo_data.py)


def check(condition, message="Check failed"):
    """Instant feedback: raises an error with a hint if the condition is False."""
    assert condition, message
    print("✅ Passed")


def check_equal(value, expected, message=None):
    assert value == expected, message or f"Expected {expected!r}, got {value!r}"
    print("✅ Passed")

> **Runtime:** in Colab, choose *Runtime → Change runtime type → T4 GPU*. With `QUICK = True` the notebook trains on a subset
> in about 3 minutes; set it to `False` for the full dataset (~15 minutes on a GPU).

## Part A — Lecture

**From pixels to patches.** Sessions 9–10 classified each pixel from its spectrum. A CNN instead looks at a **patch**
(here 64 × 64 Sentinel-2 pixels) and learns spatial features — field edges, road networks, textures — through stacked
convolution filters. Training from scratch needs a lot of data; **transfer learning** reuses a network pretrained on
ImageNet and only adapts it to our classes.

**EuroSAT** (Helber et al., 2019): 27,000 labelled Sentinel-2 patches, 10 classes (annual crop, forest, herbaceous
vegetation, highway, industrial, pasture, permanent crop, residential, river, sea/lake). We use the RGB version.

**Where are open EO training datasets today?** Radiant MLHub, used in the 2022 bootcamp, closed in 2023; its datasets
moved to **Source Cooperative** (<https://source.coop>). **TorchGeo** (<https://torchgeo.readthedocs.io>) provides
ready-made PyTorch loaders for dozens of EO datasets, including multispectral EuroSAT.

In [ ]:
import os, time, numpy as np, matplotlib.pyplot as plt
import torch, torchvision
from torch import nn
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import transforms, models

QUICK = True                                   # small subset and 1 epoch for a fast demo
CI = os.environ.get("ML4EO_CI") == "1"         # used by the course's automated tests (fake images)
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print("Device:", device)

In [ ]:
tf = transforms.Compose([transforms.ToTensor(),
                         transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
if CI:
    data = torchvision.datasets.FakeData(600, image_size=(3, 64, 64), num_classes=10, transform=tf)
    CLASS_NAMES = [f"class {i}" for i in range(10)]
else:
    data = torchvision.datasets.EuroSAT(root="eurosat", download=True, transform=tf)
    CLASS_NAMES = data.classes
if QUICK:
    data = Subset(data, torch.randperm(len(data))[:3000 if not CI else 600].tolist())
n_test = int(0.2 * len(data))
train_set, test_set = random_split(data, [len(data) - n_test, n_test], generator=torch.Generator().manual_seed(0))
train_dl = DataLoader(train_set, batch_size=64, shuffle=True, num_workers=2)
test_dl = DataLoader(test_set, batch_size=128, num_workers=2)
print(len(train_set), "training and", len(test_set), "test patches;", CLASS_NAMES)

In [ ]:
def show_batch(dl, n=8):
    x, y = next(iter(dl))
    img = x[:n].permute(0, 2, 3, 1).numpy() * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]
    fig, ax = plt.subplots(1, n, figsize=(2 * n, 2.3))
    for i in range(n):
        ax[i].imshow(img[i].clip(0, 1)); ax[i].set_title(CLASS_NAMES[y[i]], fontsize=8); ax[i].axis("off")
    plt.show()


show_batch(train_dl)

### Transfer learning with ResNet-18

In [ ]:
weights = None if CI else models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)
model.fc = nn.Linear(model.fc.in_features, len(CLASS_NAMES))     # new output layer for our 10 classes
model = model.to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)


def run_epoch(dl, train=True):
    model.train(train)
    total, correct, loss_sum = 0, 0, 0.0
    with torch.set_grad_enabled(train):
        for x, y in dl:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = loss_fn(out, y)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            loss_sum += loss.item() * len(y); correct += (out.argmax(1) == y).sum().item(); total += len(y)
    return loss_sum / total, correct / total


EPOCHS = 1 if QUICK else 5
for epoch in range(EPOCHS):
    t = time.time()
    tr_loss, tr_acc = run_epoch(train_dl, True)
    te_loss, te_acc = run_epoch(test_dl, False)
    print(f"epoch {epoch + 1}: train acc {tr_acc:.3f} | test acc {te_acc:.3f} | {time.time() - t:.0f}s")

## Part B — Lab

**Task 1 — Predictions.** Collect the true labels and predicted labels of the whole test set into two NumPy arrays
`y_true` and `y_pred`, then compute `test_accuracy`.

In [ ]:
# YOUR CODE HERE
y_true = None
y_pred = None
test_accuracy = None
raise NotImplementedError()

In [ ]:
# ✅ Check (do not modify)
check(len(y_true) == len(test_set))
check(CI or test_accuracy > 0.8, "Fine-tuned ResNet-18 should exceed 80% on EuroSAT RGB")

**Task 2 — Confusion matrix.** Plot the confusion matrix (`ConfusionMatrixDisplay.from_predictions`) with the class names and
find the most confused pair of different classes (`worst_pair`, a tuple of two names).

In [ ]:
# YOUR CODE HERE
worst_pair = None
raise NotImplementedError()

In [ ]:
# ✅ Check (do not modify)
check(isinstance(worst_pair, tuple) and worst_pair[0] != worst_pair[1])

**❓ Question —** Look at the most confused pair. Would the extra Sentinel-2 bands (red-edge, SWIR) help? How would you test it with TorchGeo's multispectral EuroSAT?

*Your answer here.*

### Responsible use
Training a CNN has a carbon cost; pretrained models inherit **geographic bias** (EuroSAT covers Europe only). Before using
a model in Quebec or South Africa, evaluate it on **local** labelled samples (Session 11).

## Summary

- CNNs learn spatial patterns from patches; transfer learning makes them practical with modest data.
- Evaluate deep models exactly like other classifiers: held-out data, confusion matrix, per-class metrics.
- Open EO datasets now live on Source Cooperative and are easy to load with TorchGeo.

**To go further:**

- Helber, P. et al. (2019). EuroSAT. *IEEE JSTARS*, 12(7), 2217–2226.
- Stewart, A. J. et al. (2022). TorchGeo: deep learning with geospatial data. *ACM SIGSPATIAL*.